In [1]:
# importações, caminhos, categorias e funções auxiliares
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

ROOT = Path(r"C:\tcc\network-ids-ml-generalization")
TRAINING_RESULTS_DIR = ROOT / "results" / "model_training"
EVALUATION_RESULTS_DIR = ROOT / "results" / "model_evaluation"
ANALYSIS_DIR = ROOT / "results" / "results_analysis"
TABLES_DIR = ANALYSIS_DIR / "tables"
BASELINE_TABLES_DIR = TABLES_DIR / "baseline"
RANDOMIZED_TABLES_DIR = TABLES_DIR / "randomized_search"
COMPARISON_TABLES_DIR = TABLES_DIR / "comparisons"
PRESENTATION_TABLES_DIR = TABLES_DIR / "presentation"
FIGURES_DIR = ANALYSIS_DIR / "figures"

for output_directory in [
    BASELINE_TABLES_DIR, RANDOMIZED_TABLES_DIR, COMPARISON_TABLES_DIR,
    PRESENTATION_TABLES_DIR, FIGURES_DIR,
]:
    output_directory.mkdir(parents=True, exist_ok=True)

MODEL_ORDER = ["decision_tree", "random_forest", "xgboost", "linear_svm", "mlp"]
METHOD_ORDER = ["baseline", "randomized_search"]
SCENARIO_ORDER = ["intra_dataset", "cross_dataset", "combined_to_unseen"]
TRAINING_SOURCE_ORDER = [
    "cicids2017",
    "unsw_nb15",
    "iot23",
    "cicids2017_unsw_nb15",
    "cicids2017_iot23",
    "unsw_nb15_iot23",
]
TEST_SOURCE_ORDER = ["cicids2017", "unsw_nb15", "iot23"]

MODEL_LABELS = {
    "decision_tree": "Árvore de decisão",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
    "linear_svm": "SVM linear",
    "mlp": "MLP",
}
MODEL_ABBREVIATIONS = {
    "decision_tree": "DT",
    "random_forest": "RF",
    "xgboost": "XGB",
    "linear_svm": "SVM",
    "mlp": "MLP",
}
METHOD_LABELS = {
    "baseline": "Baseline",
    "randomized_search": "Com hiperparâmetros",
}
SCENARIO_LABELS = {
    "intra_dataset": "Intra-dataset",
    "cross_dataset": "Cross-dataset",
    "combined_to_unseen": "Combinado",
}
SCENARIO_FILE_LABELS = {
    "intra_dataset": "intra_dataset",
    "cross_dataset": "cross_dataset",
    "combined_to_unseen": "combined",
}
SOURCE_LABELS = {
    "cicids2017": "CICIDS2017",
    "unsw_nb15": "UNSW-NB15",
    "iot23": "IoT-23",
    "cicids2017_unsw_nb15": "CICIDS2017 + UNSW-NB15",
    "cicids2017_iot23": "CICIDS2017 + IoT-23",
    "unsw_nb15_iot23": "UNSW-NB15 + IoT-23",
}
METHOD_COLORS = {
    "Baseline": "#4C78A8",
    "Com hiperparâmetros": "#F58518",
}

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


def save_table(dataframe, output_directory, filename):
    output_path = output_directory / filename
    dataframe.to_csv(output_path, index=False)
    return output_path


def format_table_for_display(dataframe):
    output = dataframe.copy()
    label_maps = {
        "model": MODEL_LABELS,
        "training_method": METHOD_LABELS,
        "scenario_type": SCENARIO_LABELS,
        "training_source": SOURCE_LABELS,
        "test_source": SOURCE_LABELS,
    }
    for column, label_map in label_maps.items():
        if column in output.columns:
            output[column] = output[column].map(label_map).fillna(output[column])
    return output


obsolete_artifacts = [
    FIGURES_DIR / "cv_f1_attack_mean_std.svg",
    FIGURES_DIR / "cv_balanced_accuracy_mean_std.svg",
    FIGURES_DIR / "model_cost_and_efficiency.svg",
    COMPARISON_TABLES_DIR / "efficiency_summary_by_model.csv",
]
for method_directory in [BASELINE_TABLES_DIR, RANDOMIZED_TABLES_DIR]:
    obsolete_artifacts.extend([
        method_directory / "best_model_by_scenario.csv",
        method_directory / "training_cost_and_model_size.csv",
        method_directory / "prediction_time_and_throughput.csv",
        method_directory / "scenario_combined_to_unseen_all_experiments.csv",
    ])
obsolete_artifacts.append(
    COMPARISON_TABLES_DIR / "baseline_vs_randomized_search_combined_to_unseen.csv"
)
for scenario_filename in SCENARIO_FILE_LABELS.values():
    obsolete_artifacts.append(
        COMPARISON_TABLES_DIR / f"best_and_worst_{scenario_filename}.csv"
    )
for obsolete_path in obsolete_artifacts:
    obsolete_path.unlink(missing_ok=True)


def finalize_figure(figure, filename, width, height):
    # gráfico interativo no notebook e também exportado em svg
    figure.update_layout(
        template="plotly_white",
        font=dict(family="Arial", size=13),
        title_x=0.5,
        margin=dict(l=80, r=40, t=130, b=80),
        legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5),
    )
    output_path = FIGURES_DIR / f"{filename}.svg"
    figure.write_image(output_path, format="svg", width=width, height=height)
    display(figure)
    return output_path

In [2]:
# carregamento dos resultados
baseline_summary = pd.read_csv(TRAINING_RESULTS_DIR / "baseline" / "summary.csv")
randomized_summary_files = [
    TRAINING_RESULTS_DIR / "randomized_search" / model_name / "summary.csv"
    for model_name in MODEL_ORDER
]
randomized_summary = pd.concat(
    [pd.read_csv(summary_path) for summary_path in randomized_summary_files],
    ignore_index=True,
)
randomized_summary["training_method"] = "randomized_search"
evaluation = pd.read_csv(EVALUATION_RESULTS_DIR / "evaluation_summary.csv")
# tabela pareada com os resultados sem e com ajuste de hiperparâmetros
comparison = pd.read_csv(EVALUATION_RESULTS_DIR / "baseline_vs_randomized_search.csv")
cv_results = pd.concat([baseline_summary, randomized_summary], ignore_index=True, sort=False)

# métricas mantidas nas tabelas de apresentação
FINAL_METRICS = [
    "accuracy",
    "balanced_accuracy",
    "precision_attack",
    "recall_attack",
    "f1_attack",
    "precision_benign",
    "recall_benign",
    "f1_benign",
    "f1_macro",
]
CV_METRICS = FINAL_METRICS.copy()

# complementação da comparação com métricas benignas e métricas do k-fold já calculadas
comparison_keys = ["model", "training_source", "test_source", "scenario_type"]
comparison_additional_columns = [
    "precision_benign", "recall_benign", "f1_benign",
    "cv_accuracy_mean", "cv_accuracy_std",
    "cv_precision_attack_mean", "cv_precision_attack_std",
    "cv_recall_attack_mean", "cv_recall_attack_std",
    "cv_precision_benign_mean", "cv_precision_benign_std",
    "cv_recall_benign_mean", "cv_recall_benign_std",
    "cv_f1_benign_mean", "cv_f1_benign_std",
    "cv_f1_macro_mean", "cv_f1_macro_std",
]
for method in METHOD_ORDER:
    method_values = evaluation[
        evaluation["training_method"].eq(method)
    ][comparison_keys + comparison_additional_columns].copy()
    method_values = method_values.rename(columns={
        column: f"{column}_{method}" for column in comparison_additional_columns
    })
    comparison = comparison.merge(method_values, on=comparison_keys, how="left", validate="one_to_one")
for metric in ["precision_benign", "recall_benign", "f1_benign"]:
    comparison[f"{metric}_delta"] = (
        comparison[f"{metric}_randomized_search"] - comparison[f"{metric}_baseline"]
    )

# verificações de segurança
assert len(baseline_summary) == 30, "O summary do baseline deve possuir 30 registros."
assert len(randomized_summary) == 30, "Os summaries do RandomizedSearch devem totalizar 30 registros."
assert len(evaluation) == 120, "A avaliação final deve possuir 120 registros."
assert len(comparison) == 60, "A comparação deve possuir 60 pares."
assert not cv_results.duplicated(["training_method", "model", "training_source"]).any()
assert not evaluation.duplicated(["training_method", "model", "training_source", "test_source"]).any()
assert not comparison.duplicated(["model", "training_source", "test_source"]).any()

required_metrics = FINAL_METRICS + ["f1_weighted"]

assert not evaluation[required_metrics].isna().any().any(), "Há métricas ausentes."
for metric in required_metrics:
    assert evaluation[metric].between(0, 1).all(), f"Valores inválidos em {metric}."

validation_summary = pd.DataFrame({
    "item": ["summary baseline", "summaries RandomizedSearch", "avaliações finais", "comparações pareadas"],
    "registros": [len(baseline_summary), len(randomized_summary), len(evaluation), len(comparison)],
    "status": "ok",
})
display(validation_summary)

,item,registros,status
0,summary baseline,30,ok
1,summaries RandomizedSearch,30,ok
2,avaliações finais,120,ok
3,comparações pareadas,60,ok


In [3]:
# tabelas completas separadas por método e por tipo de cenário
cv_table_columns = ["model", "training_method", "training_source", "params"]
for metric in CV_METRICS:
    cv_table_columns.extend([f"{metric}_mean", f"{metric}_std"])
cv_table_columns.append("elapsed_seconds")

final_table_columns = [
    "training_method", "training_source", "test_source", "model",
    "scenario_type", "params",
] + FINAL_METRICS
for metric in CV_METRICS:
    final_table_columns.extend([f"cv_{metric}_mean", f"cv_{metric}_std"])
final_table_columns.extend(["training_elapsed_seconds", "prediction_time_seconds"])

comparison_table_columns = comparison_keys.copy()
for metric in FINAL_METRICS:
    comparison_table_columns.extend([
        f"{metric}_baseline",
        f"{metric}_randomized_search",
        f"{metric}_delta",
    ])
for metric in CV_METRICS:
    comparison_table_columns.extend([
        f"cv_{metric}_mean_baseline", f"cv_{metric}_std_baseline",
        f"cv_{metric}_mean_randomized_search", f"cv_{metric}_std_randomized_search",
    ])
comparison_table_columns.extend([
    "training_elapsed_seconds_baseline", "training_elapsed_seconds_randomized_search",
    "prediction_time_seconds_baseline", "prediction_time_seconds_randomized_search",
    "params_randomized_search",
])

METHOD_TABLE_DIRECTORIES = {"baseline": BASELINE_TABLES_DIR, "randomized_search": RANDOMIZED_TABLES_DIR}
saved_tables = []
scenario_tables = {}

for method, output_directory in METHOD_TABLE_DIRECTORIES.items():
    method_cv = (
        cv_results[cv_results["training_method"].eq(method)]
        .reindex(columns=cv_table_columns)
        .rename(columns={"elapsed_seconds": "training_elapsed_seconds"})
        .sort_values(["training_source", "model"])
    )
    method_evaluation = evaluation[evaluation["training_method"].eq(method)][final_table_columns].copy()
    best_algorithm_by_experiment = (
        method_evaluation.sort_values(["f1_attack", "balanced_accuracy"], ascending=[False, False])
        .groupby(["training_source", "test_source"], as_index=False)
        .head(1)
        .sort_values(["training_source", "test_source"])
    )
    best_algorithm_filename = (
        "best_algorithm_by_experiment_baseline.csv"
        if method == "baseline"
        else "best_algorithm_by_experiment_with_hyperparameters.csv"
    )
    saved_tables.extend([
        save_table(format_table_for_display(method_cv), output_directory, "kfold_metrics.csv"),
        save_table(
            format_table_for_display(best_algorithm_by_experiment),
            output_directory,
            best_algorithm_filename,
        ),
    ])
    for scenario_type in SCENARIO_ORDER:
        scenario_table = method_evaluation[method_evaluation["scenario_type"].eq(scenario_type)].copy()
        scenario_table = format_table_for_display(scenario_table)
        scenario_tables[(method, scenario_type)] = scenario_table
        scenario_filename = SCENARIO_FILE_LABELS[scenario_type]
        saved_tables.append(save_table(
            scenario_table, output_directory, f"scenario_{scenario_filename}_all_experiments.csv"
        ))

comparison_metrics_table = comparison[comparison_table_columns].sort_values(
    ["scenario_type", "training_source", "test_source", "model"]
)
for scenario_type in SCENARIO_ORDER:
    scenario_comparison = comparison_metrics_table[comparison_metrics_table["scenario_type"].eq(scenario_type)]
    scenario_filename = SCENARIO_FILE_LABELS[scenario_type]
    saved_tables.append(save_table(
        format_table_for_display(scenario_comparison),
        COMPARISON_TABLES_DIR,
        f"baseline_vs_randomized_search_{scenario_filename}.csv",
    ))

# tabelas compactas com métricas por extenso para apresentação
def format_mean_std(mean_value, std_value):
    return f"{mean_value:.4f} ± {std_value:.4f}"


def format_delta(delta_value):
    return f"{delta_value:+.4f}"


model_order_map = {model: index for index, model in enumerate(MODEL_ORDER)}
method_order_map = {method: index for index, method in enumerate(METHOD_ORDER)}
training_source_order_map = {source: index for index, source in enumerate(TRAINING_SOURCE_ORDER)}
test_source_order_map = {source: index for index, source in enumerate(TEST_SOURCE_ORDER)}

cv_delta_keys = ["model", "training_source"]
cv_baseline = cv_results[cv_results["training_method"].eq("baseline")][
    cv_delta_keys + ["f1_attack_mean", "balanced_accuracy_mean"]
].rename(columns={
    "f1_attack_mean": "f1_attack_mean_baseline",
    "balanced_accuracy_mean": "balanced_accuracy_mean_baseline",
})
cv_adjusted = cv_results[cv_results["training_method"].eq("randomized_search")][
    cv_delta_keys + ["f1_attack_mean", "balanced_accuracy_mean"]
].rename(columns={
    "f1_attack_mean": "f1_attack_mean_adjusted",
    "balanced_accuracy_mean": "balanced_accuracy_mean_adjusted",
})
cv_deltas = cv_baseline.merge(cv_adjusted, on=cv_delta_keys, validate="one_to_one")
cv_deltas["f1_attack_mean_delta"] = (
    cv_deltas["f1_attack_mean_adjusted"] - cv_deltas["f1_attack_mean_baseline"]
)
cv_deltas["balanced_accuracy_mean_delta"] = (
    cv_deltas["balanced_accuracy_mean_adjusted"]
    - cv_deltas["balanced_accuracy_mean_baseline"]
)
cv_presentation_source = cv_results.merge(
    cv_deltas[cv_delta_keys + ["f1_attack_mean_delta", "balanced_accuracy_mean_delta"]],
    on=cv_delta_keys,
    validate="many_to_one",
)
cv_presentation_source["model_order"] = cv_presentation_source["model"].map(model_order_map)
cv_presentation_source["method_order"] = cv_presentation_source["training_method"].map(method_order_map)
cv_presentation_source["training_source_order"] = (
    cv_presentation_source["training_source"].map(training_source_order_map)
)
cv_presentation_source = cv_presentation_source.sort_values(
    ["training_source_order", "model_order", "method_order"]
)
cv_presentation = pd.DataFrame({
    "Conjunto de treinamento": cv_presentation_source["training_source"].map(SOURCE_LABELS),
    "Modelo": cv_presentation_source["model"].map(MODEL_ABBREVIATIONS),
    "Método": cv_presentation_source["training_method"].map(METHOD_LABELS),
    "F1 da classe ATTACK (média ± desvio-padrão)": [
        format_mean_std(mean_value, std_value)
        for mean_value, std_value in zip(
            cv_presentation_source["f1_attack_mean"],
            cv_presentation_source["f1_attack_std"],
        )
    ],
    "Acurácia balanceada (média ± desvio-padrão)": [
        format_mean_std(mean_value, std_value)
        for mean_value, std_value in zip(
            cv_presentation_source["balanced_accuracy_mean"],
            cv_presentation_source["balanced_accuracy_std"],
        )
    ],
    "Δ F1 da classe ATTACK": np.where(
        cv_presentation_source["training_method"].eq("randomized_search"),
        cv_presentation_source["f1_attack_mean_delta"].map(format_delta),
        "—",
    ),
    "Δ acurácia balanceada": np.where(
        cv_presentation_source["training_method"].eq("randomized_search"),
        cv_presentation_source["balanced_accuracy_mean_delta"].map(format_delta),
        "—",
    ),
})
saved_tables.append(save_table(
    cv_presentation, PRESENTATION_TABLES_DIR, "cross_validation_summary.csv"
))

evaluation_delta_columns = comparison_keys + ["f1_attack_delta", "f1_benign_delta"]
evaluation_presentation_source = evaluation.merge(
    comparison[evaluation_delta_columns],
    on=comparison_keys,
    validate="many_to_one",
)
evaluation_presentation_source["model_order"] = (
    evaluation_presentation_source["model"].map(model_order_map)
)
evaluation_presentation_source["method_order"] = (
    evaluation_presentation_source["training_method"].map(method_order_map)
)
evaluation_presentation_source["training_source_order"] = (
    evaluation_presentation_source["training_source"].map(training_source_order_map)
)
evaluation_presentation_source["test_source_order"] = (
    evaluation_presentation_source["test_source"].map(test_source_order_map)
)
presentation_evaluation_tables = {}
for scenario_type in SCENARIO_ORDER:
    scenario_source = evaluation_presentation_source[
        evaluation_presentation_source["scenario_type"].eq(scenario_type)
    ].sort_values([
        "training_source_order", "test_source_order", "model_order", "method_order"
    ])
    scenario_presentation = pd.DataFrame({
        "Conjunto de treinamento": scenario_source["training_source"].map(SOURCE_LABELS),
        "Conjunto de teste": scenario_source["test_source"].map(SOURCE_LABELS),
        "Modelo": scenario_source["model"].map(MODEL_ABBREVIATIONS),
        "Método": scenario_source["training_method"].map(METHOD_LABELS),
        "Acurácia balanceada": scenario_source["balanced_accuracy"].round(4),
        "F1 macro": scenario_source["f1_macro"].round(4),
        "Precisão da classe ATTACK": scenario_source["precision_attack"].round(4),
        "Recall da classe ATTACK": scenario_source["recall_attack"].round(4),
        "F1 da classe ATTACK": scenario_source["f1_attack"].round(4),
        "Precisão da classe BENIGN": scenario_source["precision_benign"].round(4),
        "Recall da classe BENIGN": scenario_source["recall_benign"].round(4),
        "F1 da classe BENIGN": scenario_source["f1_benign"].round(4),
        "Δ F1 da classe ATTACK": np.where(
            scenario_source["training_method"].eq("randomized_search"),
            scenario_source["f1_attack_delta"].map(format_delta),
            "—",
        ),
        "Δ F1 da classe BENIGN": np.where(
            scenario_source["training_method"].eq("randomized_search"),
            scenario_source["f1_benign_delta"].map(format_delta),
            "—",
        ),
    })
    presentation_evaluation_tables[scenario_type] = scenario_presentation
    scenario_filename = SCENARIO_FILE_LABELS[scenario_type]
    saved_tables.append(save_table(
        scenario_presentation,
        PRESENTATION_TABLES_DIR,
        f"final_evaluation_{scenario_filename}.csv",
    ))

display(pd.DataFrame({
    "tabela_salva": [path.relative_to(TABLES_DIR).as_posix() for path in saved_tables]
}))

print("\nValidação cruzada")
display(cv_presentation)
for scenario_type in SCENARIO_ORDER:
    print(f"\nAvaliação final — {SCENARIO_LABELS[scenario_type]}")
    display(presentation_evaluation_tables[scenario_type])

,tabela_salva
0,baseline/kfold_metrics.csv
1,baseline/best_algorithm_by_experiment_baseline.csv
2,baseline/scenario_intra_dataset_all_experiments.csv
3,baseline/scenario_cross_dataset_all_experiments.csv
4,baseline/scenario_combined_all_experiments.csv
5,randomized_search/kfold_metrics.csv
6,randomized_search/best_algorithm_by_experiment_with_hyperparameters.csv
7,randomized_search/scenario_intra_dataset_all_experiments.csv
8,randomized_search/scenario_cross_dataset_all_experiments.csv
9,randomized_search/scenario_combined_all_experiments.csv



Validação cruzada


,Conjunto de treinamento,Modelo,Método,F1 da classe ATTACK (média ± desvio-padrão),Acurácia balanceada (média ± desvio-padrão),Δ F1 da classe ATTACK,Δ acurácia balanceada
0,CICIDS2017,DT,Baseline,0.9846 ± 0.0006,0.9915 ± 0.0007,—,—
30,CICIDS2017,DT,Com hiperparâmetros,0.9843 ± 0.0008,0.9920 ± 0.0004,-0.0003,+0.0005
6,CICIDS2017,RF,Baseline,0.9873 ± 0.0007,0.9941 ± 0.0006,—,—
36,CICIDS2017,RF,Com hiperparâmetros,0.9875 ± 0.0008,0.9944 ± 0.0005,+0.0002,+0.0003
12,CICIDS2017,XGB,Baseline,0.9868 ± 0.0008,0.9940 ± 0.0007,—,—
42,CICIDS2017,XGB,Com hiperparâmetros,0.9870 ± 0.0010,0.9941 ± 0.0008,+0.0001,+0.0001
18,CICIDS2017,SVM,Baseline,0.6630 ± 0.0026,0.7620 ± 0.0014,—,—
48,CICIDS2017,SVM,Com hiperparâmetros,0.8498 ± 0.0028,0.9435 ± 0.0017,+0.1867,+0.1816
24,CICIDS2017,MLP,Baseline,0.9822 ± 0.0020,0.9905 ± 0.0024,—,—
54,CICIDS2017,MLP,Com hiperparâmetros,0.9852 ± 0.0013,0.9938 ± 0.0005,+0.0031,+0.0033



Avaliação final — Intra-dataset


,Conjunto de treinamento,Conjunto de teste,Modelo,Método,Acurácia balanceada,F1 macro,Precisão da classe ATTACK,Recall da classe ATTACK,F1 da classe ATTACK,Precisão da classe BENIGN,Recall da classe BENIGN,F1 da classe BENIGN,Δ F1 da classe ATTACK,Δ F1 da classe BENIGN
0,CICIDS2017,CICIDS2017,DT,Baseline,0.9921,0.9907,0.9815,0.9889,0.9852,0.9972,0.9953,0.9963,—,—
1,CICIDS2017,CICIDS2017,DT,Com hiperparâmetros,0.9921,0.9900,0.9785,0.9896,0.9840,0.9974,0.9946,0.9960,-0.0012,-0.0003
6,CICIDS2017,CICIDS2017,RF,Baseline,0.9943,0.9920,0.9811,0.9934,0.9872,0.9983,0.9952,0.9968,—,—
7,CICIDS2017,CICIDS2017,RF,Com hiperparâmetros,0.9943,0.9919,0.9809,0.9935,0.9871,0.9984,0.9952,0.9968,-0.0001,-0.0000
8,CICIDS2017,CICIDS2017,XGB,Baseline,0.9943,0.9916,0.9793,0.9940,0.9866,0.9985,0.9947,0.9966,—,—
9,CICIDS2017,CICIDS2017,XGB,Com hiperparâmetros,0.9942,0.9916,0.9795,0.9936,0.9865,0.9984,0.9948,0.9966,-0.0000,-0.0000
2,CICIDS2017,CICIDS2017,SVM,Baseline,0.7632,0.7989,0.8292,0.5550,0.6649,0.8972,0.9714,0.9329,—,—
3,CICIDS2017,CICIDS2017,SVM,Com hiperparâmetros,0.9428,0.9023,0.7597,0.9617,0.8489,0.9897,0.9240,0.9557,+0.1839,+0.0229
4,CICIDS2017,CICIDS2017,MLP,Baseline,0.9929,0.9902,0.9771,0.9916,0.9843,0.9979,0.9942,0.9960,—,—
5,CICIDS2017,CICIDS2017,MLP,Com hiperparâmetros,0.9937,0.9905,0.9766,0.9934,0.9849,0.9983,0.9940,0.9962,+0.0006,+0.0001



Avaliação final — Cross-dataset


,Conjunto de treinamento,Conjunto de teste,Modelo,Método,Acurácia balanceada,F1 macro,Precisão da classe ATTACK,Recall da classe ATTACK,F1 da classe ATTACK,Precisão da classe BENIGN,Recall da classe BENIGN,F1 da classe BENIGN,Δ F1 da classe ATTACK,Δ F1 da classe BENIGN
20,CICIDS2017,UNSW-NB15,DT,Baseline,0.4869,0.4399,0.0244,0.0029,0.0052,0.7957,0.9708,0.8746,—,—
21,CICIDS2017,UNSW-NB15,DT,Com hiperparâmetros,0.4858,0.4377,0.0051,0.0006,0.0011,0.7953,0.9710,0.8744,-0.0042,-0.0001
26,CICIDS2017,UNSW-NB15,RF,Baseline,0.4999,0.4444,0.0000,0.0000,0.0000,0.8000,0.9998,0.8888,—,—
27,CICIDS2017,UNSW-NB15,RF,Com hiperparâmetros,0.5000,0.4444,0.0000,0.0000,0.0000,0.8000,0.9999,0.8889,+0.0000,+0.0001
28,CICIDS2017,UNSW-NB15,XGB,Baseline,0.4998,0.4444,0.0000,0.0000,0.0000,0.7999,0.9996,0.8887,—,—
29,CICIDS2017,UNSW-NB15,XGB,Com hiperparâmetros,0.4999,0.4444,0.0000,0.0000,0.0000,0.8000,0.9997,0.8888,+0.0000,+0.0000
22,CICIDS2017,UNSW-NB15,SVM,Baseline,0.4987,0.4453,0.0954,0.0018,0.0036,0.7996,0.9957,0.8869,—,—
23,CICIDS2017,UNSW-NB15,SVM,Com hiperparâmetros,0.4975,0.4445,0.0601,0.0017,0.0033,0.7992,0.9934,0.8858,-0.0003,-0.0011
24,CICIDS2017,UNSW-NB15,MLP,Baseline,0.5000,0.4451,0.1975,0.0008,0.0016,0.8000,0.9992,0.8886,—,—
25,CICIDS2017,UNSW-NB15,MLP,Com hiperparâmetros,0.4991,0.4448,0.0814,0.0010,0.0021,0.7997,0.9971,0.8875,+0.0005,-0.0010



Avaliação final — Combinado


,Conjunto de treinamento,Conjunto de teste,Modelo,Método,Acurácia balanceada,F1 macro,Precisão da classe ATTACK,Recall da classe ATTACK,F1 da classe ATTACK,Precisão da classe BENIGN,Recall da classe BENIGN,F1 da classe BENIGN,Δ F1 da classe ATTACK,Δ F1 da classe BENIGN
40,CICIDS2017 + UNSW-NB15,IoT-23,DT,Baseline,0.5018,0.4548,0.2592,0.0126,0.0240,0.8006,0.9910,0.8857,—,—
41,CICIDS2017 + UNSW-NB15,IoT-23,DT,Com hiperparâmetros,0.0677,0.0909,0.0046,0.0162,0.0071,0.3265,0.1192,0.1746,-0.0169,-0.7110
46,CICIDS2017 + UNSW-NB15,IoT-23,RF,Baseline,0.4566,0.4252,0.0160,0.0060,0.0088,0.7850,0.9071,0.8416,—,—
47,CICIDS2017 + UNSW-NB15,IoT-23,RF,Com hiperparâmetros,0.0214,0.0278,0.0025,0.0097,0.0040,0.1179,0.0331,0.0517,-0.0048,-0.7899
48,CICIDS2017 + UNSW-NB15,IoT-23,XGB,Baseline,0.0201,0.0266,0.0021,0.0083,0.0034,0.1138,0.0318,0.0498,—,—
49,CICIDS2017 + UNSW-NB15,IoT-23,XGB,Com hiperparâmetros,0.0197,0.0263,0.0020,0.0079,0.0032,0.1130,0.0316,0.0494,-0.0002,-0.0004
42,CICIDS2017 + UNSW-NB15,IoT-23,SVM,Baseline,0.1081,0.1067,0.0283,0.1032,0.0444,0.3351,0.1130,0.1690,—,—
43,CICIDS2017 + UNSW-NB15,IoT-23,SVM,Com hiperparâmetros,0.1526,0.1246,0.0521,0.1960,0.0824,0.3522,0.1093,0.1668,+0.0380,-0.0022
44,CICIDS2017 + UNSW-NB15,IoT-23,MLP,Baseline,0.4158,0.1573,0.1716,0.8150,0.2835,0.2635,0.0165,0.0311,—,—
45,CICIDS2017 + UNSW-NB15,IoT-23,MLP,Com hiperparâmetros,0.4117,0.1514,0.1704,0.8121,0.2816,0.1923,0.0112,0.0211,-0.0019,-0.0100


In [4]:
# seleção de casos candidatos para discussão posterior
comparison_case_columns = [
    "model", "training_source", "test_source", "scenario_type",
    "f1_attack_baseline", "f1_attack_randomized_search", "f1_attack_delta",
    "balanced_accuracy_delta", "precision_attack_delta", "recall_attack_delta",
]


def label_comparison_cases(rows, highlight):
    output = rows[comparison_case_columns].copy()
    output.insert(0, "destaque", highlight)
    return output


comparison_with_abs_delta = comparison.assign(f1_attack_abs_delta=comparison["f1_attack_delta"].abs())
comparison_cases = pd.concat([
    label_comparison_cases(
        comparison.nlargest(3, "f1_attack_delta"),
        "Maior aumento no F1 de ataque",
    ),
    label_comparison_cases(
        comparison.nsmallest(3, "f1_attack_delta"),
        "Maior redução no F1 de ataque",
    ),
    label_comparison_cases(
        comparison_with_abs_delta.nsmallest(3, "f1_attack_abs_delta"),
        "Menor alteração no F1 de ataque",
    ),
    label_comparison_cases(
        comparison.nlargest(3, "precision_attack_delta"),
        "Maior aumento na precisão de ataque",
    ),
    label_comparison_cases(
        comparison.nlargest(3, "recall_attack_delta"),
        "Maior aumento no recall de ataque",
    ),
], ignore_index=True)

performance_case_columns = [
    "model", "training_method", "training_source", "test_source", "scenario_type",
] + FINAL_METRICS


def label_performance_cases(rows, highlight):
    output = rows[performance_case_columns].copy()
    output.insert(0, "destaque", highlight)
    return output


evaluation_with_accuracy_f1_difference = evaluation.assign(
    accuracy_f1_attack_difference=evaluation["accuracy"] - evaluation["f1_attack"]
)
best_per_scenario_type = (
    evaluation.sort_values(
        ["scenario_type", "f1_attack", "balanced_accuracy"],
        ascending=[True, False, False],
    )
    .groupby("scenario_type", as_index=False)
    .head(1)
)
worst_per_scenario_type = (
    evaluation.sort_values(
        ["scenario_type", "f1_attack", "balanced_accuracy"],
        ascending=[True, True, True],
    )
    .groupby("scenario_type", as_index=False)
    .head(1)
)
performance_extremes = pd.concat([
    label_performance_cases(best_per_scenario_type, "Melhor resultado por cenário"),
    label_performance_cases(worst_per_scenario_type, "Pior resultado por cenário"),
], ignore_index=True)
performance_metric_discrepancies = label_performance_cases(
    evaluation_with_accuracy_f1_difference.nlargest(5, "accuracy_f1_attack_difference"),
    "Acurácia alta e F1 de ataque baixo",
)

# contagens calculadas sobre os 60 experimentos pareados, sem agregação prévia
effect_rows = []
for scenario_type in SCENARIO_ORDER:
    scenario_deltas = comparison.loc[
        comparison["scenario_type"].eq(scenario_type), "f1_attack_delta"
    ]
    effect_rows.append({
        "scenario_type": scenario_type,
        "experiments": len(scenario_deltas),
        "improved": int(scenario_deltas.gt(0).sum()),
        "worsened": int(scenario_deltas.lt(0).sum()),
        "unchanged": int(scenario_deltas.eq(0).sum()),
        "f1_attack_delta_mean": scenario_deltas.mean(),
        "f1_attack_delta_median": scenario_deltas.median(),
    })
effect_by_scenario = pd.DataFrame(effect_rows)

saved_tables.append(save_table(
    format_table_for_display(comparison_cases),
    COMPARISON_TABLES_DIR,
    "cases_baseline_vs_randomized_search.csv",
))
saved_tables.append(save_table(
    format_table_for_display(performance_metric_discrepancies),
    COMPARISON_TABLES_DIR,
    "performance_metric_discrepancies.csv",
))
saved_tables.append(save_table(
    format_table_for_display(effect_by_scenario),
    COMPARISON_TABLES_DIR,
    "hyperparameter_effect_by_scenario.csv",
))

# um melhor e um pior resultado são salvos separadamente por cenário
for scenario_type in SCENARIO_ORDER:
    scenario_extremes = performance_extremes[
        performance_extremes["scenario_type"].eq(scenario_type)
    ]
    saved_tables.append(save_table(
        format_table_for_display(scenario_extremes),
        COMPARISON_TABLES_DIR,
        f"performance_extremes_{SCENARIO_FILE_LABELS[scenario_type]}.csv",
    ))

display(format_table_for_display(comparison_cases))
display(format_table_for_display(performance_extremes))
display(format_table_for_display(performance_metric_discrepancies))
display(format_table_for_display(effect_by_scenario))

,destaque,model,training_source,test_source,scenario_type,f1_attack_baseline,f1_attack_randomized_search,f1_attack_delta,balanced_accuracy_delta,precision_attack_delta,recall_attack_delta
0,Maior aumento no F1 de ataque,SVM linear,CICIDS2017,IoT-23,Cross-dataset,0.044105,0.358254,0.314148,0.444643,0.190306,0.893584
1,Maior aumento no F1 de ataque,SVM linear,CICIDS2017,CICIDS2017,Intra-dataset,0.664926,0.848853,0.183928,0.179632,-0.069519,0.406738
2,Maior aumento no F1 de ataque,SVM linear,CICIDS2017 + IoT-23,UNSW-NB15,Combinado,0.025785,0.156293,0.130508,0.035794,0.261059,0.084270
3,Maior redução no F1 de ataque,Árvore de decisão,IoT-23,CICIDS2017,Cross-dataset,0.550507,0.025045,-0.525462,-0.233397,-0.523883,-0.501263
4,Maior redução no F1 de ataque,MLP,UNSW-NB15,CICIDS2017,Cross-dataset,0.450475,0.332993,-0.117482,-0.095163,-0.177978,0.019173
5,Maior redução no F1 de ataque,MLP,IoT-23,UNSW-NB15,Cross-dataset,0.119335,0.012886,-0.106449,0.009537,0.260111,-0.083478
6,Menor alteração no F1 de ataque,Random Forest,CICIDS2017,UNSW-NB15,Cross-dataset,0.000000,0.000000,0.000000,0.000056,0.000000,0.000000
7,Menor alteração no F1 de ataque,XGBoost,CICIDS2017,UNSW-NB15,Cross-dataset,0.000000,0.000000,0.000000,0.000043,0.000000,0.000000
8,Menor alteração no F1 de ataque,SVM linear,IoT-23,IoT-23,Intra-dataset,0.993126,0.993102,-0.000025,-0.000006,-0.000049,0.000000
9,Maior aumento na precisão de ataque,Random Forest,IoT-23,UNSW-NB15,Cross-dataset,0.177697,0.121307,-0.056390,0.018039,0.284894,-0.080753


,destaque,model,training_method,training_source,test_source,scenario_type,accuracy,balanced_accuracy,precision_attack,recall_attack,f1_attack,precision_benign,recall_benign,f1_benign,f1_macro
0,Melhor resultado por cenário,MLP,Baseline,UNSW-NB15 + IoT-23,CICIDS2017,Combinado,0.800436,0.714051,0.500958,0.570077,0.533287,0.888679,0.858026,0.873084,0.703185
1,Melhor resultado por cenário,SVM linear,Baseline,UNSW-NB15,CICIDS2017,Cross-dataset,0.842596,0.707462,0.641703,0.482239,0.550659,0.878131,0.932685,0.904586,0.727623
2,Melhor resultado por cenário,XGBoost,Baseline,IoT-23,IoT-23,Intra-dataset,0.998851,0.997238,0.999701,0.994550,0.997119,0.998639,0.999926,0.999282,0.998201
3,Pior resultado por cenário,XGBoost,Baseline,CICIDS2017 + IoT-23,UNSW-NB15,Combinado,0.799088,0.499579,0.074074,0.000396,0.000788,0.799865,0.998761,0.888316,0.444552
4,Pior resultado por cenário,MLP,Com hiperparâmetros,IoT-23,CICIDS2017,Cross-dataset,0.786852,0.491782,0.000000,0.000000,0.000000,0.797335,0.983565,0.880713,0.440356
5,Pior resultado por cenário,SVM linear,Baseline,CICIDS2017,CICIDS2017,Intra-dataset,0.888135,0.763197,0.829225,0.554967,0.664926,0.897238,0.971427,0.932860,0.798893


,destaque,model,training_method,training_source,test_source,scenario_type,accuracy,balanced_accuracy,precision_attack,recall_attack,f1_attack,precision_benign,recall_benign,f1_benign,f1_macro
27,Acurácia alta e F1 de ataque baixo,Random Forest,Com hiperparâmetros,CICIDS2017,UNSW-NB15,Cross-dataset,0.799950,0.499969,0.000000,0.000000,0.00000,0.799990,0.999938,0.888858,0.444429
26,Acurácia alta e F1 de ataque baixo,Random Forest,Baseline,CICIDS2017,UNSW-NB15,Cross-dataset,0.799861,0.499913,0.000000,0.000000,0.00000,0.799972,0.999827,0.888803,0.444402
29,Acurácia alta e F1 de ataque baixo,XGBoost,Com hiperparâmetros,CICIDS2017,UNSW-NB15,Cross-dataset,0.799782,0.499864,0.000000,0.000000,0.00000,0.799956,0.999728,0.888754,0.444377
28,Acurácia alta e F1 de ataque baixo,XGBoost,Baseline,CICIDS2017,UNSW-NB15,Cross-dataset,0.799713,0.499820,0.000000,0.000000,0.00000,0.799943,0.999641,0.888711,0.444356
39,Acurácia alta e F1 de ataque baixo,XGBoost,Com hiperparâmetros,CICIDS2017 + IoT-23,UNSW-NB15,Combinado,0.799832,0.500062,0.257143,0.000446,0.00089,0.800020,0.999678,0.888774,0.444832


,scenario_type,experiments,improved,worsened,unchanged,f1_attack_delta_mean,f1_attack_delta_median
0,Intra-dataset,15,7,8,0,0.013168,-0.000025
1,Cross-dataset,30,10,18,2,-0.019852,-0.000403
2,Combinado,15,6,9,0,0.015845,-0.001895


In [5]:
# qualidade média por algoritmo e tipo de cenário
def plot_metric_by_scenario(metric, metric_label, filename):
    plot_data = (
        evaluation.groupby(
            ["scenario_type", "model", "training_method"], as_index=False
        )
        .agg(
            metric_mean=(metric, "mean"),
            metric_min=(metric, "min"),
            metric_max=(metric, "max"),
            experiments=(metric, "size"),
        )
    )
    plot_data["distance_to_min"] = plot_data["metric_mean"] - plot_data["metric_min"]
    plot_data["distance_to_max"] = plot_data["metric_max"] - plot_data["metric_mean"]
    plot_data["model_label"] = plot_data["model"].map(MODEL_LABELS)
    plot_data["method_label"] = plot_data["training_method"].map(METHOD_LABELS)
    plot_data["scenario_label"] = plot_data["scenario_type"].map(SCENARIO_LABELS)
    figure = px.bar(
        plot_data,
        x="model_label",
        y="metric_mean",
        color="method_label",
        facet_col="scenario_label",
        barmode="group",
        error_y="distance_to_max",
        error_y_minus="distance_to_min",
        color_discrete_map=METHOD_COLORS,
        category_orders={
            "model_label": [MODEL_LABELS[model] for model in MODEL_ORDER],
            "method_label": [METHOD_LABELS[method] for method in METHOD_ORDER],
            "scenario_label": [SCENARIO_LABELS[scenario] for scenario in SCENARIO_ORDER],
        },
        custom_data=["experiments", "metric_min", "metric_max"],
        labels={"model_label": "Algoritmo", "metric_mean": metric_label, "method_label": "Método"},
        title=(
            f"{metric_label} médio nas avaliações finais por algoritmo e tipo de cenário"
            "<br><sup>Cada barra é a média; os traços indicam o menor e o maior resultado observado</sup>"
        ),
    )
    figure.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split("=")[-1]))
    figure.update_traces(
        hovertemplate=(
            "Algoritmo: %{x}<br>Média: %{y:.4f}<br>"
            "Mínimo: %{customdata[1]:.4f}<br>Máximo: %{customdata[2]:.4f}<br>"
            "Experimentos: %{customdata[0]}<extra></extra>"
        ),
        error_y=dict(color="#333333", thickness=1.5, width=5),
    )
    figure.update_yaxes(range=[0, 1.05], matches="y")
    figure.update_xaxes(tickangle=-25)
    finalize_figure(figure, filename, width=1500, height=520)


plot_metric_by_scenario("f1_attack", "F1 da classe ATTACK", "f1_attack_by_scenario_type")
plot_metric_by_scenario("balanced_accuracy", "Acurácia balanceada", "balanced_accuracy_by_scenario_type")
plot_metric_by_scenario("precision_attack", "Precisão da classe ATTACK", "precision_attack_by_scenario_type")
plot_metric_by_scenario("recall_attack", "Recall da classe ATTACK", "recall_attack_by_scenario_type")

In [6]:
# médias e desvios-padrão dos cinco folds permanecem nas tabelas
# os gráficos de k-fold foram omitidos porque não acrescentavam clareza à apresentação

In [7]:
# mapas de calor da generalização separados por método
def plot_generalization_heatmap(method):
    pivot = (
        evaluation[evaluation["training_method"].eq(method)]
        .groupby(["training_source", "test_source"])["f1_attack"]
        .mean()
        .unstack("test_source")
        .reindex(index=TRAINING_SOURCE_ORDER, columns=TEST_SOURCE_ORDER)
        .rename(index=SOURCE_LABELS, columns=SOURCE_LABELS)
    )
    figure = px.imshow(
        pivot,
        text_auto=".3f",
        color_continuous_scale="YlGnBu",
        zmin=0,
        zmax=1,
        aspect="auto",
        labels={"x": "Dataset de teste", "y": "Origem do treinamento", "color": "F1 médio da classe ATTACK"},
        title=(
            f"Generalização média entre datasets — {METHOD_LABELS[method]}"
            "<br><sup>Cada célula é a média dos cinco algoritmos</sup>"
        ),
    )
    finalize_figure(figure, f"generalization_f1_attack_{method}", width=900, height=720)


for method in METHOD_ORDER:
    plot_generalization_heatmap(method)

delta_pivot = (
    comparison.groupby(["training_source", "test_source"])["f1_attack_delta"]
    .mean()
    .unstack("test_source")
    .reindex(index=TRAINING_SOURCE_ORDER, columns=TEST_SOURCE_ORDER)
    .rename(index=SOURCE_LABELS, columns=SOURCE_LABELS)
)
delta_limit = max(float(np.nanmax(np.abs(delta_pivot.to_numpy(dtype=float)))), 0.01)
delta_figure = px.imshow(
    delta_pivot,
    text_auto=".3f",
    color_continuous_scale="RdBu",
    zmin=-delta_limit,
    zmax=delta_limit,
    color_continuous_midpoint=0,
    aspect="auto",
    labels={"x": "Dataset de teste", "y": "Origem do treinamento", "color": "Variação do F1 da classe ATTACK"},
    title=(
        "Variação média do F1 de ataque após o ajuste de hiperparâmetros"
        "<br><sup>Cada célula é a média dos cinco algoritmos</sup>"
    ),
)
finalize_figure(delta_figure, "generalization_f1_attack_delta_heatmap", width=900, height=720)

WindowsPath('C:/tcc/network-ids-ml-generalization/results/results_analysis/figures/generalization_f1_attack_delta_heatmap.svg')

In [8]:
# variação do f1 de ataque em todos os experimentos pareados
delta_plot_data = comparison.copy()
delta_plot_data["model_label"] = delta_plot_data["model"].map(MODEL_LABELS)
delta_plot_data["scenario_label"] = delta_plot_data["scenario_type"].map(SCENARIO_LABELS)
delta_plot_data["training_source_label"] = delta_plot_data["training_source"].map(SOURCE_LABELS)
delta_plot_data["test_source_label"] = delta_plot_data["test_source"].map(SOURCE_LABELS)
delta_plot_data["direction"] = np.select(
    [delta_plot_data["f1_attack_delta"].gt(0), delta_plot_data["f1_attack_delta"].lt(0)],
    ["Aumento", "Redução"],
    default="Sem alteração",
)
delta_all_figure = px.scatter(
    delta_plot_data,
    x="f1_attack_delta",
    y="model_label",
    color="direction",
    facet_col="scenario_label",
    color_discrete_map={"Aumento": "#2E7D32", "Redução": "#C62828", "Sem alteração": "#666666"},
    category_orders={
        "model_label": [MODEL_LABELS[model] for model in MODEL_ORDER],
        "scenario_label": [SCENARIO_LABELS[scenario] for scenario in SCENARIO_ORDER],
    },
    hover_data={
        "training_source_label": True,
        "test_source_label": True,
        "model_label": False,
        "scenario_label": False,
    },
    labels={
        "f1_attack_delta": "Variação do F1 da classe ATTACK",
        "model_label": "Algoritmo",
        "direction": "Resultado",
        "training_source_label": "Treinamento",
        "test_source_label": "Teste",
    },
    title="Efeito do ajuste de hiperparâmetros nos 60 experimentos pareados",
)
delta_all_figure.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split("=")[-1]))
delta_all_figure.add_vline(x=0, line_width=1, line_color="#444444")
delta_all_figure.update_traces(marker=dict(size=9, opacity=0.8))
finalize_figure(delta_all_figure, "f1_attack_delta_all_experiments", width=1500, height=620)

# maiores aumentos e reduções individuais
largest_f1_attack_changes = pd.concat([
    comparison.nsmallest(8, "f1_attack_delta"),
    comparison.nlargest(8, "f1_attack_delta"),
], ignore_index=True).drop_duplicates(["model", "training_source", "test_source"])
largest_f1_attack_changes = largest_f1_attack_changes.sort_values("f1_attack_delta")
largest_f1_attack_changes["experiment_label"] = largest_f1_attack_changes.apply(
    lambda row: (
        f"{MODEL_LABELS[row['model']]} | {SOURCE_LABELS[row['training_source']]} → {SOURCE_LABELS[row['test_source']]}"
    ),
    axis=1,
)
largest_f1_attack_changes["direction"] = np.where(
    largest_f1_attack_changes["f1_attack_delta"].ge(0), "Aumento", "Redução"
)
delta_figure = px.bar(
    largest_f1_attack_changes,
    x="f1_attack_delta",
    y="experiment_label",
    color="direction",
    orientation="h",
    color_discrete_map={"Aumento": "#2E7D32", "Redução": "#C62828"},
    labels={"f1_attack_delta": "Variação do F1 da classe ATTACK", "experiment_label": "Modelo e cenário", "direction": "Resultado"},
    title="Maiores aumentos e reduções após o ajuste de hiperparâmetros",
)
delta_figure.add_vline(x=0, line_width=1, line_color="#444444")
finalize_figure(delta_figure, "f1_attack_largest_changes", width=1200, height=850)

# melhor e pior resultado de f1 de ataque em cada tipo de cenário
extremes_plot_data = performance_extremes.copy()
extremes_plot_data["scenario_label"] = extremes_plot_data["scenario_type"].map(SCENARIO_LABELS)
extremes_plot_data["model_label"] = extremes_plot_data["model"].map(MODEL_LABELS)
extremes_plot_data["method_label"] = extremes_plot_data["training_method"].map(METHOD_LABELS)
extremes_plot_data["training_source_label"] = extremes_plot_data["training_source"].map(SOURCE_LABELS)
extremes_plot_data["test_source_label"] = extremes_plot_data["test_source"].map(SOURCE_LABELS)
extremes_plot_data["point_label"] = extremes_plot_data.apply(
    lambda row: f"{MODEL_ABBREVIATIONS[row['model']]} · {row['f1_attack']:.3f}",
    axis=1,
)

extremes_figure = go.Figure()
for scenario_type in SCENARIO_ORDER:
    scenario_rows = extremes_plot_data[extremes_plot_data["scenario_type"].eq(scenario_type)]
    extremes_figure.add_trace(go.Scatter(
        x=[scenario_rows["f1_attack"].min(), scenario_rows["f1_attack"].max()],
        y=[SCENARIO_LABELS[scenario_type], SCENARIO_LABELS[scenario_type]],
        mode="lines",
        line=dict(color="#B0B0B0", width=3),
        hoverinfo="skip",
        showlegend=False,
    ))

extreme_styles = {
    "Melhor resultado por cenário": ("Melhor resultado", "#2E7D32", "top center"),
    "Pior resultado por cenário": ("Pior resultado", "#C62828", "bottom center"),
}
for highlight, (result_label, color, text_position) in extreme_styles.items():
    result_rows = extremes_plot_data[extremes_plot_data["destaque"].eq(highlight)]
    custom_data = result_rows[[
        "model_label", "method_label", "training_source_label", "test_source_label"
    ]].to_numpy()
    extremes_figure.add_trace(go.Scatter(
        x=result_rows["f1_attack"],
        y=result_rows["scenario_label"],
        mode="markers+text",
        text=result_rows["point_label"],
        textposition=text_position,
        name=result_label,
        marker=dict(color=color, size=15),
        customdata=custom_data,
        hovertemplate=(
            "Cenário: %{y}<br>Resultado: " + result_label + "<br>"
            "Algoritmo: %{customdata[0]}<br>Método: %{customdata[1]}<br>"
            "Treinamento: %{customdata[2]}<br>Teste: %{customdata[3]}<br>"
            "F1 da classe ATTACK: %{x:.4f}<extra></extra>"
        ),
    ))

extremes_figure.update_xaxes(
    title_text="F1 da classe ATTACK",
    range=[0, 1.05],
)
extremes_figure.update_yaxes(
    title_text="Tipo de cenário",
    categoryorder="array",
    categoryarray=[SCENARIO_LABELS[scenario] for scenario in reversed(SCENARIO_ORDER)],
)
extremes_figure.update_layout(
    title="Melhor e pior resultado por tipo de cenário",
)
finalize_figure(extremes_figure, "f1_attack_best_worst_by_scenario", width=1200, height=560)

# quantidade de melhorias e reduções após o ajuste de hiperparâmetros
effect_plot_data = effect_by_scenario.melt(
    id_vars=["scenario_type", "experiments"],
    value_vars=["improved", "worsened", "unchanged"],
    var_name="effect",
    value_name="count",
)
effect_plot_data["scenario_label"] = effect_plot_data["scenario_type"].map(SCENARIO_LABELS)
effect_plot_data["effect_label"] = effect_plot_data["effect"].map({
    "improved": "Melhoria",
    "worsened": "Redução",
    "unchanged": "Sem alteração",
})
effect_count_figure = px.bar(
    effect_plot_data,
    x="scenario_label",
    y="count",
    color="effect_label",
    barmode="group",
    text="count",
    custom_data=["experiments"],
    color_discrete_map={
        "Melhoria": "#2E7D32",
        "Redução": "#C62828",
        "Sem alteração": "#666666",
    },
    category_orders={
        "scenario_label": [SCENARIO_LABELS[scenario] for scenario in SCENARIO_ORDER],
        "effect_label": ["Melhoria", "Redução", "Sem alteração"],
    },
    labels={
        "scenario_label": "Tipo de cenário",
        "count": "Quantidade de experimentos",
        "effect_label": "Efeito no F1 da classe ATTACK",
    },
    title=(
        "Efeito do ajuste de hiperparâmetros por tipo de cenário"
        "<br><sup>Intra-dataset: 15 pares; cross-dataset: 30 pares; combinado: 15 pares</sup>"
    ),
)
effect_count_figure.update_traces(
    textposition="outside",
    hovertemplate=(
        "Cenário: %{x}<br>Resultado: %{fullData.name}<br>"
        "Quantidade: %{y}<br>Total de pares no cenário: %{customdata[0]}<extra></extra>"
    ),
)
effect_count_figure.update_yaxes(rangemode="tozero", dtick=5)
finalize_figure(
    effect_count_figure,
    "hyperparameter_effect_counts_by_scenario",
    width=1100,
    height=560,
)

WindowsPath('C:/tcc/network-ids-ml-generalization/results/results_analysis/figures/hyperparameter_effect_counts_by_scenario.svg')

In [9]:
# resumo dos artefatos gerados pela execução atual
generated_tables = sorted({path.relative_to(TABLES_DIR).as_posix() for path in saved_tables})
generated_figures = sorted(path.name for path in FIGURES_DIR.glob("*.svg"))
print(f"Tabelas salvas: {len(generated_tables)}")
print(f"Gráficos salvos em SVG: {len(generated_figures)}")
display(pd.DataFrame({"tabelas": generated_tables}))
display(pd.DataFrame({"graficos_svg": generated_figures}))

Tabelas salvas: 23
Gráficos salvos em SVG: 11


,tabelas
0,baseline/best_algorithm_by_experiment_baseline.csv
1,baseline/kfold_metrics.csv
2,baseline/scenario_combined_all_experiments.csv
3,baseline/scenario_cross_dataset_all_experiments.csv
4,baseline/scenario_intra_dataset_all_experiments.csv
5,comparisons/baseline_vs_randomized_search_combined.csv
6,comparisons/baseline_vs_randomized_search_cross_dataset.csv
7,comparisons/baseline_vs_randomized_search_intra_dataset.csv
8,comparisons/cases_baseline_vs_randomized_search.csv
9,comparisons/hyperparameter_effect_by_scenario.csv


,graficos_svg
0,balanced_accuracy_by_scenario_type.svg
1,f1_attack_best_worst_by_scenario.svg
2,f1_attack_by_scenario_type.svg
3,f1_attack_delta_all_experiments.svg
4,f1_attack_largest_changes.svg
5,generalization_f1_attack_baseline.svg
6,generalization_f1_attack_delta_heatmap.svg
7,generalization_f1_attack_randomized_search.svg
8,hyperparameter_effect_counts_by_scenario.svg
9,precision_attack_by_scenario_type.svg
